# 👀 Notebook 05 — Giving the AI Eyes: Deep Q-Networks (DQN)

**Series**: RL Notebook Series · Act II — Value-Based Methods · Post 5 of 15

---

## What You'll Learn

In Notebook 04, we used a massive spreadsheet (a numpy array) to store exactly one $Q$-value for every possible integer state. 

What happens when the state isn't a simple integer grid, but a **continuous array of decimal numbers**? (Like the joint angles of a robot, the coordinates of a spaceship, or the pixel values of a screen).

We can't build a table with infinite rows. Instead, we have to teach a Neural Network to *look* at a state and *predict* the $Q$-values for that state.

By the end, you'll understand:

1. **Function Approximation** — swapping the Q-table for a PyTorch Neural Network.
2. **Continuous State Spaces** — introducing the classic `CartPole` environment.
3. **Catastrophic Forgetting** — why RL breaks neural networks out of the box.
4. **Experience Replay** — how to give an AI a "memory" to stabilize learning.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import deque

%matplotlib inline
plt.rcParams.update({'figure.figsize': (10, 6), 'axes.grid': True, 'grid.alpha': 0.3})

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

## 1. The Environment: CartPole

We drop our toy grid world and introduce Gymnasium's `CartPole-v1`. 

A pole is attached by an un-actuated joint to a cart, which moves along a frictionless track. The system is controlled by applying a force of +1 or -1 to the cart (pushing it left or right). 

- **Goal:** Keep the pole from falling over for as long as possible.
- **State Space:** 4 continuous numbers: `[Cart Position, Cart Velocity, Pole Angle, Pole Angular Velocity]`
- **Action Space:** 2 discrete choices: `0` (push left), `1` (push right)
- **Reward:** +1 for every step the pole remains upright.

Because the state space is continuous, a tabular approach like Notebook 04 is physically impossible.

In [ ]:
env = gym.make('CartPole-v1')
state, info = env.reset(seed=42)

print("Action space:", env.action_space)
print("State space:", env.observation_space)
print("Initial state:", state)
print(f"\n(Cart Pos: {state[0]:.4f}, Cart Vel: {state[1]:.4f}, Pole Angle: {state[2]:.4f}, Pole AngVel: {state[3]:.4f})"
)

## 2. Enter the Neural Network: $Q_\theta(s, a)$

Instead of `Q[state, action]`, we will feed the state into a Neural Network parametrized by weights $\theta$. The network will have one output neuron for each possible action. The values of these neurons are the predicted $Q$-values.

Because we are only passing in 4 numbers, we don't need a massive deep learning architecture — a tiny Multi-Layer Perceptron (MLP) with two hidden layers of 64 neurons will do.

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(QNetwork, self).__init__()
        # Input layer -> Hidden 1
        self.fc1 = nn.Linear(state_dim, 64)
        # Hidden 1 -> Hidden 2
        self.fc2 = nn.Linear(64, 64)
        # Hidden 2 -> Output (Q-values)
        self.fc3 = nn.Linear(64, action_dim)

    def forward(self, state):
        """
        Forward pass.
        Input: Tensor of shape (batch_size, state_dim)
        Returns: Tensor of shape (batch_size, action_dim) containing Q-values
        """
        # ============================================================
        # TODO: Implement the forward pass.
        # Pass the state through fc1, apply ReLU.
        # Pass through fc2, apply ReLU.
        # Pass through fc3 (NO activation function on the last layer!)
        # ============================================================
        raise NotImplementedError

<details>
<summary>🔍 <b>Hint</b> (click to reveal)</summary>

Use `F.relu()` for activations.

    x = F.relu(self.fc1(state))
    x = F.relu(self.fc2(x))
    return self.fc3(x)

We don't use activation on the final layer because Q-values can legitimately be any real number (negative or extremely large).

</details>

In [ ]:
# Let's test the untrained network
state_dim = env.observation_space.shape[0]  # 4
action_dim = env.action_space.n             # 2

q_net = QNetwork(state_dim, action_dim)

# Convert numpy array state to PyTorch tensor (and add a batch dimension)
state_tensor = torch.FloatTensor(state).unsqueeze(0)

with torch.no_grad():
    q_vals = q_net(state_tensor)

print(f"Untrained Q-values for start state: {q_vals.numpy()[0]}")
print(f"Greedy action: {torch.argmax(q_vals).item()}")

## 3. The Unlearning Problem (Catastrophic Interference)

In normal deep learning (like classifying dogs vs. cats), we shuffle the dataset so the network sees a random mix of dogs and cats.

In RL, the data arrives sequentially. Imagine the CartPole falling to the left. For 50 consecutive frames, it tries to learn to push left. The network weights shift dramatically. Then the pole swings right. The network suddenly sees 50 frames of "push right", and its neural pathways completely obliterate everything it just learned about pushing left.

Sequential data breaks neural networks. We have to break the correlation between consecutive frames.

## 4. The Replay Buffer (Memory)

The fix (introduced by DeepMind in 2013) is incredibly elegant: **give the agent a short-term memory.**

Instead of updating the network immediately after every step, we simply store the transition `(state, action, reward, next_state, done)` into a massive `ReplayBuffer`.

When it's time to train, we draw a random handful (`batch_size`) of memories from the *entire* buffer. The network might see one frame from 5 minutes ago of the pole falling left, mixed with a frame from just now of the pole falling right. The data is "shuffled" again, and the network can learn stably.

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity=10000):
        # deque automatically throws away the oldest memories when it reaches capacity
        self.buffer = deque(maxlen=capacity)

    def add(self, state, action, reward, next_state, done):
        """Save a transition to memory."""
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        """
        Draw a random batch of transitions.
        Returns them as PyTorch tensors ready for the neural network.
        """
        # ============================================================
        # TODO: Implement random sampling from self.buffer.
        # 1. Select `batch_size` random elements from self.buffer.
        #    (Hint: random.sample(population, k))
        # 2. Unpack them into lists of states, actions, rewards,
        #    next_states, and dones.
        # ============================================================
        raise NotImplementedError

        # Convert to arrays -> tensors
        states = torch.FloatTensor(np.array(states))
        actions = torch.LongTensor(np.array(actions)).unsqueeze(1)
        rewards = torch.FloatTensor(np.array(rewards)).unsqueeze(1)
        next_states = torch.FloatTensor(np.array(next_states))
        dones = torch.FloatTensor(np.array(dones)).unsqueeze(1)

        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.buffer)

<details>
<summary>🔍 <b>Hint</b> (click to reveal)</summary>

Use `random.sample` to grab a random subset without replacement:

    batch = random.sample(self.buffer, batch_size)

Then you can unpack the list of tuples using standard zip tricks:

    states, actions, rewards, next_states, dones = zip(*batch)

</details>

## 5. Training the Deep Q-Network

The math is identical to Notebook 04, just translated into deep learning operations.

**1. Compute the TD Target:** $\text{Target} = R + \gamma \max_{a'} Q_\theta(s', a')$
**2. Compute the Loss:** $\text{MSE}(\text{Prediction}, \text{Target})$
**3. Backpropagate:** Tell PyTorch to update the weights via `optimizer.step()`

*(Warning: this cell will take roughly 30 to 60 seconds to run on a normal CPU!)*

In [ ]:
def train_dqn(env_name='CartPole-v1', num_episodes=500, batch_size=64, gamma=0.99, epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995):
    env = gym.make(env_name)

    q_net = QNetwork(env.observation_space.shape[0], env.action_space.n)
    # Standard Adam optimizer
    optimizer = optim.Adam(q_net.parameters(), lr=1e-3)
    buffer = ReplayBuffer(capacity=10000)

    epsilon = epsilon_start
    rewards_history = []

    for ep in range(num_episodes):
        state, _ = env.reset()
        total_reward = 0
        done = False
        truncated = False

        while not (done or truncated):
            # --- 1. Epsilon-Greedy Action Selection ---
            if random.random() < epsilon:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    state_t = torch.FloatTensor(state).unsqueeze(0)
                    q_vals = q_net(state_t)
                    action = torch.argmax(q_vals).item()

            # --- 2. Take Step and Store in Memory ---
            next_state, reward, done, truncated, _ = env.step(action)
            buffer.add(state, action, reward, next_state, done or truncated)
            state = next_state
            total_reward += reward

            # --- 3. Learn from a random batch of memories ---
            if len(buffer) > batch_size:
                states_b, actions_b, rewards_b, next_states_b, dones_b = buffer.sample(batch_size)

                # Current Q-values (Prediction)
                # q_net(states_b) outputs all actions. `gather` extracts just the Q-value for the action we actually took.
                current_q = q_net(states_b).gather(1, actions_b)

                # ============================================================
                # TODO: Compute the target Q-values and the MSE loss.
                #
                # 1. Pass `next_states_b` through the network to get Q-values.
                # 2. Take the max over actions (dim=1). Remember to .unsqueeze(1)
                #    so it aligns properly with the rewards_b tensor.
                # 3. Compute target: R + gamma * max_next_Q * (1 - done)
                # 4. Use F.mse_loss to compute the error between current_q and target_q.
                #
                # NOTE: Target calculation MUST be inside a `with torch.no_grad():`
                # block so gradients don't flow backward into the target!
                # ============================================================
                raise NotImplementedError

                # Backpropagation update
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        # --- 4. End of Episode Bookkeeping ---
        epsilon = max(epsilon_end, epsilon * epsilon_decay)
        rewards_history.append(total_reward)

        # Print progress
        if (ep + 1) % 50 == 0:
            avg_reward = np.mean(rewards_history[-50:])
            print(f"Episode {ep+1:3d} | Epsilon: {epsilon:.2f} | Average Reward (last 50): {avg_reward:.1f}")

            if avg_reward >= 450.0:
                print("Solved!")
                break

    return q_net, rewards_history

<details>
<summary>🔍 <b>Hint</b> (click to reveal)</summary>

Here is the standard PyTorch implementation of the DQN loss:

    with torch.no_grad():
        # max(1)[0] extracts the maximum value. unsqueeze(1) makes it a column vector.
        max_next_q = q_net(next_states_b).max(1)[0].unsqueeze(1)
        target_q = rewards_b + gamma * max_next_q * (1 - dones_b)

    loss = F.mse_loss(current_q, target_q)

We multiply by `(1 - dones_b)` because if the episode ends at the next state, the expected future reward is 0.

</details>

In [ ]:
trained_net, rewards = train_dqn(num_episodes=500)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rewards, color='#6366f1', alpha=0.3)

window = 20
if len(rewards) >= window:
    smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
    ax.plot(np.arange(window-1, len(rewards)), smoothed, color='#6366f1', linewidth=2, label=f'{window}-ep moving avg')

ax.set_title("DQN Training on CartPole-v1")
ax.set_xlabel("Episode")
ax.set_ylabel("Total Reward (Duration)")
ax.axhline(500, color='gray', linestyle='--', label="Game Won (Max Reward)")
ax.legend()
plt.show()

## 6. The Moving Target Problem

Look at the learning curve above. You might see it shoot up to maximum reward (500) and then suddenly collapse back down to 100 or 200, bouncing around erratically.

Why does this happen? Isn't training a neural network always a bit noisy?

If you train a Convolutional Neural Network to recognize dogs, the accuracy curve might wiggle up and down a little (e.g., jumping from 92% to 94% to 91%) because of the random batches of data. But the ground truth target (whether a picture is actually a dog or a cat) never changes.

In Basic DQN, the network is trying to predict a target that is **generated by the network itself** (the Bellman equation uses $Q_\theta(s')$ to update $Q_\theta(s)$). 

If the network accidentally overestimates the value of a state by 10% because of normal neural network noise, that overestimated value is immediately used as the "ground truth target" to update the states that lead to it. This causes a devastating domino effect where the network continuously chases its own tail, and performance can catastrophically collapse from a perfect 500 reward down to 15 reward in just a few episodes.

Why is basic DQN so unstable?

Look at the update rule again:
$$ \text{Loss} = \text{MSE}\big( \underbrace{R + \gamma \max_{a'} Q_\theta(s', a')}_{\text{Target based on } \theta} , \underbrace{Q_\theta(s, a)}_{\text{Prediction based on } \theta} \big) $$

We are computing the target using the **exact same neural network weights** $\theta$ that we are currently updating.

Imagine a dog chasing its own tail. Every time the network updates its guess to get closer to the target, the target itself shifts because the network's understanding of the next state changed too. This rapidly leads to positive feedback loops where the network hallucinate infinitely large Q-values.

### What's coming next

In **Notebook 06**, we apply two legendary fixes that stabilised Deep RL:

1. **Target Networks** — using a frozen, delayed copy of the network to compute the target, stopping the dog from chasing its tail.
2. **Double DQN** — decoupling the action *selection* from the action *evaluation* to prevent overestimating Q-values.

These are the exact tricks that finally allowed DeepMind to conquer Atari games.